# Doublets_4 Clustering - Spleen 46 only

Publication source copy for the Doublets 2 and 4 sequencing analyses. Outputs and execution counts are cleared. See `../VALIDATION_REPORT.md` for the separate execution results and remaining limitations.

Run from the `analysis/` directory after supplying the bundled inputs. Final figure sections explicitly load preserved manuscript checkpoints; they are distinct from fresh clustering.

In [ ]:
from pathlib import Path
Path("../results/Doublets_4_Clustering_Sp46").mkdir(parents=True, exist_ok=True)
from threadpoolctl import threadpool_limits
threadpool_limits(limits=12, user_api="blas")


In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
from anndata import read_h5ad
from anndata import read_text
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from matplotlib.pyplot import rc_context
import anndata
import seaborn as sns
import regex as re
import os

In [ ]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=200, dpi_save=300)
sc.settings.n_jobs = 30
pd.options.display.max_columns = None
pd.options.display.max_rows = None

sc.logging.print_header()

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Import h5ad with processed data and BCR info

In [ ]:
path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'
adata = sc.read_h5ad(path_data+'Doublets4_7600cells_GeneSymbols_preprocessed_2xDoubletCounts.h5ad')

#the line below fixes an error that results when trying to run sc.tl.rank_genes_groups(adata,'leiden',method='wilcoxon')
#error is: KeyError: 'base'
adata.uns['log1p']["base"] = None
adata.obs.head()

# Filter interactions by imaging quality

In [ ]:
#filter our real doublets with low score; keep all artificial doublets and singlets
list_not_doubletscored = ['ArtDoub', 'Sing']
adata = adata[(adata.obs['sortedtype'].isin(list_not_doubletscored)) | (adata.obs['score']=='1')]
adata.shape

# Remove spheromers

In [ ]:
adata = adata[~((adata.obs['sortedpop'].str.contains('-HA')) | (adata.obs['sortedpop'].str.contains('-PRTN3')))]
adata.shape

# Subset 1 donor

In [ ]:
# Materialize the donor subset before PCA. The AnnData view and copy
# have identical expression values but can yield different numerical
# PCA results; the explicit copy reproduces the reference partition.
adata = adata[adata.obs['SampleID']=='Spleen46'].copy()

adata.shape

In [ ]:
Counter(adata.obs['sortedpop'])

# PCA

In [ ]:
#compute pca
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.pl.pca_scatter(adata, color=['SampleID'])#, save='_all_ages_droplet_facs_tissue.pdf')
sc.pl.pca(adata, color=['384plate', 'sortedtype', 'sortedpop'], wspace=0.5)
sc.pl.pca_scatter(adata, color=['n_counts', 'n_genes'])

# Cluster

In [ ]:
# Compute neighborhood graph followed by UMAP
sc.pp.neighbors(adata)
sc.tl.umap(adata)

#recommended method: Leiden graph clustering
sc.tl.leiden(adata,resolution=0.8)

In [ ]:
#plot
sc.pl.umap(adata, color=['leiden'], legend_fontsize=8)

sc.pl.umap(adata, color=['leiden','SampleID','Stimulation','sortedtype','sortedpop'], legend_fontsize=8)
df_grouped = adata.obs.groupby('leiden')['sortedpop'].value_counts(normalize=True).unstack()
df_grouped.plot.bar(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata.obs.groupby('sortedpop')['leiden'].value_counts(normalize=True).unstack()
df_grouped.plot.bar(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

# Rename leiden clusters

In [ ]:
sc.pl.umap(adata, color=['leiden','sortedpop','sortedtype'], legend_fontsize=8, wspace=0.4)

In [ ]:
#rename
new_cluster_names = ['No Intx 1',
                     'Tfh - LZ GC B Intx',
                     'Treg/Tfr',
                     'T - Plasmablast',
                     'Tfh',
                     'B',
                     'Treg-B Cytotoxic No Intx',
                     'No Intx 2', 
                     'T - Plasmablast Cycling',
                     'Tfh - AID+SOX5+ GC B Intx']
adata.rename_categories('leiden', new_cluster_names)

adata = adata[~adata.obs['leiden'].isin(['Unknown1','Unknown2'])]

In [ ]:
adata_real = adata[adata.obs['sortedtype']=='RealDoub']
adata_real.shape

# Plots

In [ ]:
sc.pl.umap(adata, color=['leiden','sortedpop','sortedtype'], legend_fontsize=8, wspace=0.4)
df_grouped = adata.obs.groupby('leiden')['sortedpop'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata.obs.groupby('sortedpop')['leiden'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata_real.obs.groupby('leiden')['Stimulation'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata_real.obs.groupby('Stimulation')['leiden'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata_real.obs.groupby('SampleID')['Stimulation'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

# Find DEGs

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden', method='logreg', tie_correct=True, pts=True, key_added='logreg')
cluster_genes = pd.DataFrame(adata.uns['logreg']['names'])
cluster_genes.head(100)

# Merge BCR data with RNA data

In [ ]:
df_bcr = pd.read_csv(path_data+'Doublets4_TCR_BCR_productive_completeVDJ_UniqueClones.txt', sep='\t')
print('df_bcr ',df_bcr.shape)

### Drop nan IGH ###
df_bcr.dropna(subset=['locus_h','locus_l'],inplace=True)
print('Drop nan IGH ',df_bcr.shape)
df_bcr.reset_index(inplace=True, drop=True)

### Drop rows that have an 'N' in their sequence alignment ####
df_bcr = df_bcr[~(df_bcr['sequence_alignment_h'].str.contains('N'))]
df_bcr.reset_index(inplace=True, drop=True)
df_bcr = df_bcr[~(df_bcr['sequence_alignment_l'].str.contains('N'))]
df_bcr.reset_index(inplace=True, drop=True)
print('Drop rows that have an N in their sequence alignment', df_bcr.shape)
df_bcr.head()

#### Merge adata with bcr #####
adata.obs = adata.obs.merge(df_bcr, on=['barcode'], how='left')
adata.obs.head()

In [ ]:
df_grouped = adata.obs.groupby('leiden')['c_call_h'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

# Import TCR data and merge with RNA+BCR

In [ ]:
#import
path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'
df_ir = pd.read_csv(path_data+'Doublets4_TCR_BCR_productive_completeVDJ.txt', sep='\t')

#separate heavy and light chain
df_tcr_b = df_ir[df_ir['locus']=='TRB']
df_tcr_a = df_ir[df_ir['locus']=='TRA']

#keep only the first occurence of each TRB per donor
print(len(df_tcr_b))
df_tcr_b.sort_values(by=['v_score'],inplace=True,ascending=False)
df_tcr_b.drop_duplicates(subset=['cell_id'],inplace=True)
print(len(df_tcr_b))

#keep only the first occurence of each IgKL per donor
print(len(df_tcr_a))
df_tcr_a.sort_values(by=['v_score'],inplace=True,ascending=False)
df_tcr_a.drop_duplicates(subset=['cell_id'],inplace=True)
print(len(df_tcr_a))

#merge heavy and light chain
df_tcr = df_tcr_b.merge(df_tcr_a, on=['cell_id','barcode'], how='left', suffixes=('_b', '_a'))
print(df_tcr.shape)

### Drop nan TCR ###
df_tcr.dropna(subset=['locus_b','locus_a'],inplace=True)
print('Drop nan TCR ',df_tcr.shape)
df_tcr.reset_index(inplace=True, drop=True)

## Drop rows that have an 'N' in their sequence alignment ####

#drop columns that cause problem when saving adata (and is not useful)
df_tcr.dropna(axis=1, how='all',inplace=True)

#### Merge adata with bcr #####
adata.obs = adata.obs.merge(df_tcr, on=['barcode','cell_id'], how='left')
adata.obs.head()

# Add TCR and BCR expansion numbers

In [ ]:
###### Calculate clonal expansion by amino acid
adata.obs['junction_aa_h_count'] = [list(adata.obs['junction_aa_h']).count(x) if pd.notna(x) else np.nan for x in list(adata.obs['junction_aa_h'])]
adata.obs['junction_aa_h_expanded'] = ['Y' if x>1 else 'N' for x in list(adata.obs['junction_aa_h_count'])]

adata.obs['junction_aa_b_count'] = [list(adata.obs['junction_aa_b']).count(x) if pd.notna(x) else np.nan for x in list(adata.obs['junction_aa_b'])]
adata.obs['junction_aa_b_expanded'] = ['Y' if x>1 else 'N' for x in list(adata.obs['junction_aa_b_count'])]

In [ ]:
sc.pl.umap(adata, color=['leiden','sortedpop','junction_aa_h_count','junction_aa_h_expanded','junction_aa_b_count','junction_aa_b_expanded','c_call_h'], legend_fontsize=8, cmap='YlGnBu')

# Save

In [ ]:
#save processed data
adata.write_h5ad(path_data+'Doublets_SP46_1961cells_processed_2xDoubletCounts_BCR_TCR.h5ad')

# Manuscript figures 202410

In [ ]:
# Use the preserved manuscript object for the published figure layout.
path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'
adata = sc.read_h5ad('../data/reference_checkpoints/Doublets_SP46_1961cells_processed_2xDoubletCounts_BCR_TCR.h5ad')

adata.uns['log1p']["base"] = None
adata

# #rename leiden clusters
new_cluster_names = ['No Intx 1', 
                     'Th1/Tfh - MZB Intx',
                     'Treg/Tfr',
                     'T - Plasmablast Intx',
                     'Tfh',
                     'B',
                     'Cytotoxic Treg - B No-Intx',
                     'No Intx 2',
                     'T - Plasmablast Cycling Intx',
                     'Tfh - GCB Intx'
                     ]
adata.rename_categories('leiden', new_cluster_names)

#edit names
dict_leiden_old_to_new = {'No Intx 1':'T-B non-interacting',
                            'No Intx 2':'T-B non-interacting',
                            'Cytotoxic Treg - B No-Intx':'Cytotoxic Treg-B non-interacting',
                            'Th1/Tfh - MZB Intx':'Th1/Tfh-MZB',
                            'Tfh':'Tfh',
                            'B':'B',
                            'Treg/Tfr':'Treg/Tfr',
                            'Tfh - GCB Intx':'Tfh-GCB',
                            'T - Plasmablast Intx':'T-Plasmablast',
                            'T - Plasmablast Cycling Intx':'T-Plasmablast Cycling'}
adata.obs['leiden'] = (
    adata.obs['leiden']
    .map(dict_leiden_old_to_new)
    .astype('category')
)

#rename sortedpop categories

dict_sortedpop_old_to_new = {'CD25pTB': 'CD25+ T-B',
         'CD25nTB': 'CD25- T-B', 
         'ArtDoub-Tfr-B': 'Artificial Tfr-B', 
         'ArtDoub-Treg-B': 'Artificial Treg-B', 
         'ArtDoub-Tfh-B': 'Artificial Tfh-B', 
         'Treg': 'Treg',
         'Tfr': 'Tfr',
         'Tfh': 'Tfh',
         'B': 'B'}
adata.obs['sortedpop'] = (
    adata.obs['sortedpop']
    .map(dict_sortedpop_old_to_new)
    .astype('category')
)

In [ ]:
Counter(adata.obs['leiden'])
set(adata.obs['leiden'])

In [ ]:
path_figures = '../results/Doublets_4_Clustering_Sp46/'
sc.settings.figdir = path_figures
sc.set_figure_params(dpi=300, dpi_save=600)

leiden_order_1 = ['T-Plasmablast Cycling',
                  'T-Plasmablast',
                  'Th1/Tfh-MZB',
                  'Cytotoxic Treg-B non-interacting',
                  'Tfh-GCB',
                  'T-B non-interacting',
                  'B',
                  'Treg/Tfr',
                  'Tfh']
leiden_order_2 = leiden_order_1[::-1]

sortedpop_order_1 = ['CD25- T-B',
                     'CD25+ T-B',
                     'Artificial Tfr-B',
                     'Artificial Treg-B',
                     'Artificial Tfh-B',
                     'Treg',
                     'Tfr',
                     'Tfh',
                     'B']
sortedpop_order_2 = sortedpop_order_1[::-1]

In [ ]:
#color palette
leiden_palette_2 = [
    "#A1D99B",  # Light Green
    "#FED976",  # Yellow
    
    "#FD8D3C",  # Orange

    "#969696",  # Medium Gray
    "#31A354",  # Green
    "#B2182B",  # Dark Red
    
    "#3182BD",  # Blue
    "#BC80BD",  # Purple

    "#4B0082",  # Dark Purple
]

In [ ]:
#all 
adata_plot_leiden = adata.copy()
adata_plot_leiden.obs['leiden'] = adata_plot_leiden.obs['leiden'].cat.set_categories(leiden_order_2, ordered=True)
sc.pl.umap(adata_plot_leiden, color='leiden', legend_fontsize=12, frameon=False, title='', palette=leiden_palette_2, save='_Fig_S3_SP46_leiden.pdf')

#singlets
adata_plot_leiden_singlet = adata.copy()
sortedpop_order_2_singlets = sortedpop_order_2[0:4]
adata_plot_leiden_singlet.obs['leiden'] = [x if y in sortedpop_order_2_singlets else 'none' for x,y in zip(adata_plot_leiden_singlet.obs['leiden'],adata_plot_leiden_singlet.obs['sortedpop'])] #mask to color leiden clusters by sorted population
adata_plot_leiden_singlet.obs['leiden'] = adata_plot_leiden_singlet.obs['leiden'].astype('category') #stop error
adata_plot_leiden_singlet.obs['leiden'] = adata_plot_leiden_singlet.obs['leiden'].cat.set_categories(leiden_order_2, ordered=True)
sc.pl.umap(adata_plot_leiden_singlet, color='leiden', legend_fontsize=12, frameon=False, title='Singlets', palette=leiden_palette_2, na_color='whitesmoke', legend_loc = None, save='_Fig_S3_SP46_leiden_Singlets.pdf')

#artificial doublets
adata_plot_leiden_artdoub = adata.copy()
sortedpop_order_2_artdoub = sortedpop_order_2[4:7]
adata_plot_leiden_artdoub.obs['leiden'] = [x if y in sortedpop_order_2_artdoub else 'none' for x,y in zip(adata_plot_leiden_artdoub.obs['leiden'],adata_plot_leiden_artdoub.obs['sortedpop'])] #mask to color leiden clusters by sorted population
adata_plot_leiden_artdoub.obs['leiden'] = adata_plot_leiden_artdoub.obs['leiden'].astype('category') #stop error
adata_plot_leiden_artdoub.obs['leiden'] = adata_plot_leiden_artdoub.obs['leiden'].cat.set_categories(leiden_order_2, ordered=True)
sc.pl.umap(adata_plot_leiden_artdoub, color='leiden', legend_fontsize=12, frameon=False, title='Non-interacting Controls', palette=leiden_palette_2, na_color='whitesmoke', legend_loc = None, save='_Fig_S3_SP46_leiden_ArtDoub.pdf')

#real doublets
adata_plot_leiden_realdoub = adata.copy()
sortedpop_order_2_realdoub = sortedpop_order_2[7:9]
adata_plot_leiden_realdoub.obs['leiden'] = [x if y in sortedpop_order_2_realdoub else 'none' for x,y in zip(adata_plot_leiden_realdoub.obs['leiden'],adata_plot_leiden_realdoub.obs['sortedpop'])] #mask to color leiden clusters by sorted population
adata_plot_leiden_realdoub.obs['leiden'] = adata_plot_leiden_realdoub.obs['leiden'].astype('category') #stop error
adata_plot_leiden_realdoub.obs['leiden'] = adata_plot_leiden_realdoub.obs['leiden'].cat.set_categories(leiden_order_2, ordered=True)
sc.pl.umap(adata_plot_leiden_realdoub, color='leiden', legend_fontsize=12, frameon=False, title='Contacting Pairs', palette=leiden_palette_2, na_color='whitesmoke', legend_loc = None, save='_Fig_S3_SP46_leiden_RealDoub.pdf')

In [ ]:
Counter(adata.obs['leiden'])

In [ ]:
## Plot distribution of sorted populations
df_grouped = adata.obs.groupby('sortedpop')['leiden'].value_counts(normalize=True).unstack()
df_grouped = df_grouped.reindex(sortedpop_order_2) #order rows
df_grouped = df_grouped[leiden_order_2] #order columns

#singlet

#singlet
df_grouped.loc[sortedpop_order_2[0:4]].plot.barh(stacked=True, color=leiden_palette_2, legend=False)
plt.grid(False)
plt.ylabel('')
sns.despine()
plt.savefig(path_figures+'barplot_Fig_S3_SP46_singlet.pdf', bbox_inches='tight')
plt.show()

# artificial doublet
df_grouped.loc[sortedpop_order_2[4:7]].plot.barh(stacked=True, color=leiden_palette_2, legend=False)
plt.grid(False)
plt.ylabel('')
sns.despine()
plt.savefig(path_figures+'barplot_Fig_S3_SP46_ArtDoub.pdf', bbox_inches='tight')
plt.show()

# real doublet
df_grouped.loc[sortedpop_order_1[0:2]].plot.barh(stacked=True, color=leiden_palette_2, legend=False)
plt.grid(False)
plt.ylabel('')
sns.despine()
plt.savefig(path_figures+'barplot_Fig_S3_SP46_RealDoub.pdf', bbox_inches='tight')

plt.show()

In [ ]:
list_plot_genes = ['CD3E', 'MS4A1', 'FOXP3', 'IL2RA', 'TNFRSF18', 'PRF1', 'GZMA', 'GZMB', 'BCL6', 'AICDA', 'CD1C', 'TBX21', 'IL12A', 'IL12B', 'CXCR3', 'NRN1']

with plt.rc_context():  # Use this to set figure params like size and dpi
    #scaled
    sc.pl.dotplot(adata, list_plot_genes, 'leiden', dendrogram=False, cmap='Reds', standard_scale='var', colorbar_title='column scaled\nexpression',show=True, swap_axes=False, categories_order=leiden_order_2, save='Fig_S3_SP46.pdf')

In [ ]:
list_plot_genes = ['CD3E','CD4','CD8A','MS4A1',#lineage
                   'FOXP3','IL2RA','TNFRSF18','PDCD1',#Treg
                   'PRF1','GZMA','GZMB', #Cytotoxic
                   'CD40LG','CXCR5','BCL6','MIR155HG','CCL22','EBI3','BCL2A1', #LZ
                   'AICDA','SOX5','ITGAX',#GC/ ABC B cells
                   'CD1C','CXCL9','CXCL10','CXCR3','ITGAV','ITGB3','CXCL13', #Th1-MZB
                   'MZB1','JCHAIN','SDC1','CD38','MYC','TFAP4','TNFRSF13B','IRF4','MKI67','CIITA','HLA-DRB1','TNFSF8','ITGB2','ITGB3','HIF1A','MYC','CCND1','FOS','JUN','CD40LG','CD40' #plasmablast
                   
                  ]

with plt.rc_context():  # Use this to set figure params like size and dpi
    #scaled
    sc.pl.dotplot(adata, list_plot_genes, 'leiden', dendrogram=False, cmap='Reds', standard_scale='var', colorbar_title='column scaled\nexpression',show=True, swap_axes=True, categories_order=leiden_order_2)

In [ ]:
# Plot interactions by stimulation condition - ONLY realdoub 
adata_realdoublet = adata[adata.obs['sortedtype'].isin(['RealDoub'])]

df_grouped = adata_realdoublet.obs.groupby('Stimulation')['leiden'].value_counts(normalize=True).unstack()
df_grouped = df_grouped[leiden_order_2[3:]] #order columns

stimulation_palette = [
    
    "#B3DDF2",  # IGHD*00,IGHM*00 (similar light blue)
    "#FF7F00",  # IGHM*00 (distinct orange)
    "#6A3D9A",  # IGHG3*00 (dark purple)
    "#9E83AB",  # IGHG2*00 (medium purple)
    "#CAB2D6",  # IGHG1*00 (light purple)

     "#0B3D0B","#4C9B4C" #green
]

df_grouped.plot.barh(stacked=True, color=leiden_palette_2[3:])

plt.legend(
    bbox_to_anchor=(1.05, 0.5),  # Position the legend to the right of the figure
    loc="center left",           # Align the legend to the left vertically centered
    borderaxespad=0,
    frameon=False,
    fontsize=14,
    ncol=1                       # Single column
)
plt.grid(False)

plt.ylabel('')
plt.xlabel('% of cluster')

plt.show()

In [ ]:
# Plot IG isotype distribution
df_grouped = adata.obs.groupby('leiden')['c_call_h'].value_counts(normalize=True).unstack()

df_grouped = df_grouped.rename(columns={'IGHD*00': 'IGHD', 
                                        'IGHM*00,IGHD*00': 'IGHD,IGHM',
                                        'IGHM*00':'IGHM', 
                                        'IGHG1*00':'IGHG1', 
                                        'IGHG2*00':'IGHG2', 
                                        'IGHG3*00':'IGHG3',
                                        'IGHG4*00':'IGHG4',
                                        'IGHA1*00':'IGHA1', 
                                        'IGHA2*00':'IGHA2',})
df_grouped = df_grouped[['IGHD', 'IGHD,IGHM', 'IGHM', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4', 'IGHA1', 'IGHA2']] #order columns

isotype_palette = [
    "#A6CEE3",  # IGHD*00 (light blue)
    "#B3DDF2",  # IGHD*00,IGHM*00 (similar light blue)
    "#FF7F00",  # IGHM*00 (distinct orange)
    "#6A3D9A",  # IGHG3*00 (dark purple)
    "#9E83AB",  # IGHG2*00 (medium purple)
    "#CAB2D6",  # IGHG1*00 (light purple)
    
    "#FFEDA0",   # IGHA2*00 (light yellow-orange)
     "#0B3D0B","#4C9B4C" #green
    
]

df_grouped.loc[leiden_order_1[:-3]].plot.barh(stacked=True, color=isotype_palette)

plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.ylabel('Annotated Leiden Cluster')
plt.xlabel('% of cluster')

plt.show()

In [ ]:
list_plot_genes = ['CD3E', 'CD4', 'CD8A', 'MS4A1', 'BCL6', 'MYC', 'MIR155HG', 'CCL22', 'EBI3', 'BCL2A1', 'FAS', 'AICDA', 'SOX5', 'ITGAX', 'ZEB2']
with plt.rc_context():
    sc.pl.dotplot(adata, list_plot_genes, 'leiden', dendrogram=False, cmap='Reds', standard_scale='var', colorbar_title='column scaled\nexpression', show=True, swap_axes=False, categories_order=leiden_order_2, save='dotplot_SP46_nonSpheromer.pdf')